# 24 — g:Profiler online enrichment validation

**Historical executed notebook:**  
`22b_online_enrichment_validation_HVG8000_r0_6_v2_no_pip_direct_api.ipynb`

**Repository notebook:**  
`24_gprofiler_validation.ipynb`

## Historical audit result

Unlike the curated-ORA issue corrected in Notebook 23, the historical g:Profiler
workflow was already direction-aware.

It constructed:

- five malignant-state lists using only genes annotated as upregulated in each state;
- two directional lists for each of the six condition contrasts using the sign of
  `log2FC_A_vs_B`.

Therefore, this notebook preserves the historical g:Profiler gene-list semantics.

The historical executed notebook also contained an environment-state dependency:
`RUN_ENRICHR` was referenced without being defined in the notebook source. Its executed
Enrichr result tables were empty, whereas g:Profiler produced the supporting online
validation used by this branch. The repository workflow therefore makes the intended
submitted analysis explicit and runs **g:Profiler only**.

## Reproducibility note

The gene-list construction is deterministic and is hard-checked against historical
inventory checkpoints. g:Profiler itself is an online service backed by an evolving
annotation database, so present-day term counts and p-values are recorded as a new API
snapshot and are **not** forced to equal the historical executed output.

Historical executed g:Profiler snapshot:
- 17 submitted directional gene lists;
- 1 transient timeout for `state_marker__AC_like_reactive_malignant__up`;
- 1,669 returned result rows;
- 160 top-term rows;
- 43 biological keyword-summary rows;
- 15 biological category-count rows.

These historical result counts are audit references, not assertions for the current API.

In [1]:
# =========================
# 0. Imports, paths, and fixed historical settings
# =========================
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import os
import time
import urllib.request

import numpy as np
import pandas as pd
from IPython.display import display


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "online_enrichment_validation_HVG8000_r0_6_gprofiler_only"

STATE_SIG_TSV = (
    METADATA_DIR
    / "malignant_state_pseudobulk_DE_state_vs_rest_significant_HVG8000_r0_6.tsv"
)

COND_SIG_TSV = (
    METADATA_DIR
    / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv"
)

MAX_GENES_PER_LIST = 300
MIN_GENES_PER_LIST = 10

REQUEST_SLEEP_SEC = 1.0
REQUEST_TIMEOUT_SEC = 180
MAX_RETRIES = 3

GPROFILER_URL = "https://biit.cs.ut.ee/gprofiler/api/gost/profile/"
GPROFILER_SOURCES = ["GO:BP", "REAC", "KEGG", "WP"]

EXPECTED_INPUT_ROWS = {
    "state_significant": 20800,
    "condition_significant": 27004,
}

EXPECTED_HISTORICAL_LIST_SIZES = {
    "condition_overall__GBM_vs_LGG__GBM_up": 300,
    "condition_overall__GBM_vs_LGG__LGG_up": 300,
    "condition_overall__GBM_vs_ndGBM__GBM_up": 300,
    "condition_overall__GBM_vs_ndGBM__ndGBM_up": 300,
    "condition_overall__GBM_vs_rGBM__GBM_up": 300,
    "condition_overall__GBM_vs_rGBM__rGBM_up": 300,
    "condition_overall__LGG_vs_ndGBM__LGG_up": 300,
    "condition_overall__LGG_vs_ndGBM__ndGBM_up": 300,
    "condition_overall__LGG_vs_rGBM__LGG_up": 300,
    "condition_overall__LGG_vs_rGBM__rGBM_up": 300,
    "condition_overall__ndGBM_vs_rGBM__ndGBM_up": 196,
    "condition_overall__ndGBM_vs_rGBM__rGBM_up": 300,
    "state_marker__AC_like_reactive_malignant__up": 300,
    "state_marker__NPC_neuronal_like_malignant__up": 300,
    "state_marker__OPC_NPC_like_malignant__up": 300,
    "state_marker__cycling_malignant_candidate__up": 300,
    "state_marker__hypoxia_stress_malignant__up": 300,
}

HISTORICAL_API_SNAPSHOT = {
    "submitted_gene_lists": 17,
    "successful_gene_lists": 16,
    "gprofiler_result_rows": 1669,
    "gprofiler_top_term_rows": 160,
    "gprofiler_error_rows": 1,
    "biological_keyword_summary_rows": 43,
    "biological_category_count_rows": 15,
    "historical_failed_gene_list": "state_marker__AC_like_reactive_malignant__up",
    "historical_failure": "TimeoutError('The read operation timed out')",
}


def write_tsv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_name(path.name + ".tmp")
    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=False,
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )

    return path


for required in (
    STATE_SIG_TSV,
    COND_SIG_TSV,
):
    if not required.exists():
        raise FileNotFoundError(
            f"Required canonical DE table not found: {required}"
        )


print(
    "Historical notebook mapping:",
    "22b_online_enrichment_validation_HVG8000_r0_6_v2_no_pip_direct_api.ipynb "
    "-> 24_gprofiler_validation.ipynb",
)

print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)

print(
    "Analysis mode: g:Profiler only; historical directional gene-list semantics preserved."
)


Historical notebook mapping: 22b_online_enrichment_validation_HVG8000_r0_6_v2_no_pip_direct_api.ipynb -> 24_gprofiler_validation.ipynb
PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
Analysis mode: g:Profiler only; historical directional gene-list semantics preserved.


In [2]:
# =========================
# 1. Load canonical DE inputs and reproduce historical directional gene lists
# =========================
state_sig = pd.read_csv(
    STATE_SIG_TSV,
    sep="\t",
    low_memory=False,
)

cond_sig = pd.read_csv(
    COND_SIG_TSV,
    sep="\t",
    low_memory=False,
)


if len(state_sig) != EXPECTED_INPUT_ROWS["state_significant"]:
    raise RuntimeError(
        f"State-DE row checkpoint failed: "
        f"{len(state_sig)} != {EXPECTED_INPUT_ROWS['state_significant']}"
    )


if len(cond_sig) != EXPECTED_INPUT_ROWS["condition_significant"]:
    raise RuntimeError(
        f"Condition-DE row checkpoint failed: "
        f"{len(cond_sig)} != {EXPECTED_INPUT_ROWS['condition_significant']}"
    )


required_state_columns = {
    "malignant_state",
    "gene",
    "direction",
    "padj_fdr_bh",
}

required_condition_columns = {
    "contrast",
    "condition_A",
    "condition_B",
    "gene",
    "log2FC_A_vs_B",
    "padj_welch",
}


if not required_state_columns.issubset(state_sig.columns):
    raise RuntimeError(
        "Canonical state-DE columns are incomplete: "
        + str(
            sorted(
                required_state_columns
                - set(state_sig.columns)
            )
        )
    )


if not required_condition_columns.issubset(cond_sig.columns):
    raise RuntimeError(
        "Canonical condition-DE columns are incomplete: "
        + str(
            sorted(
                required_condition_columns
                - set(cond_sig.columns)
            )
        )
    )


def clean_gene(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value.lower() in {
        "",
        "nan",
        "none",
    }:
        return None

    return value


def unique_genes(values):
    output = []
    seen = set()

    for value in values:
        gene = clean_gene(value)

        if (
            gene
            and gene not in seen
        ):
            output.append(gene)
            seen.add(gene)

    return output


gene_lists = {}
inventory_rows = []


# Historical malignant-state logic:
# sort by state and FDR, then use only rows whose upstream direction contains "up".
ordered_state = state_sig.sort_values(
    [
        "malignant_state",
        "padj_fdr_bh",
    ],
    kind="mergesort",
)


for state, sub in ordered_state.groupby(
    "malignant_state",
    sort=True,
):
    up = sub.loc[
        sub[
            "direction"
        ]
        .astype(str)
        .str.contains(
            "up",
            case=False,
            na=False,
        )
    ]

    genes = unique_genes(
        up[
            "gene"
        ].head(
            MAX_GENES_PER_LIST
        )
    )

    name = (
        f"state_marker__{state}__up"
    )

    if len(genes) >= MIN_GENES_PER_LIST:
        gene_lists[
            name
        ] = genes

        inventory_rows.append(
            {
                "gene_list": name,
                "source": "malignant_state_marker_DE",
                "group": state,
                "direction": "up",
                "n_genes": len(genes),
            }
        )


# Historical condition logic:
# one A-up and one B-up list per contrast based on sign(log2FC_A_vs_B).
ordered_condition = cond_sig.sort_values(
    [
        "contrast",
        "padj_welch",
    ],
    kind="mergesort",
)


for contrast, sub in ordered_condition.groupby(
    "contrast",
    sort=True,
):
    condition_a = str(
        sub[
            "condition_A"
        ].iloc[
            0
        ]
    )

    condition_b = str(
        sub[
            "condition_B"
        ].iloc[
            0
        ]
    )

    fold_change = pd.to_numeric(
        sub[
            "log2FC_A_vs_B"
        ],
        errors="coerce",
    )

    directional_groups = [
        (
            f"{condition_a}_up",
            sub.loc[
                fold_change > 0
            ],
        ),
        (
            f"{condition_b}_up",
            sub.loc[
                fold_change < 0
            ],
        ),
    ]

    for direction, selected in directional_groups:
        genes = unique_genes(
            selected[
                "gene"
            ].head(
                MAX_GENES_PER_LIST
            )
        )

        name = (
            f"condition_overall__{contrast}__{direction}"
        )

        if len(genes) >= MIN_GENES_PER_LIST:
            gene_lists[
                name
            ] = genes

            inventory_rows.append(
                {
                    "gene_list": name,
                    "source": "condition_overall_malignant_DE",
                    "group": contrast,
                    "direction": direction,
                    "n_genes": len(genes),
                }
            )


inventory = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        [
            "source",
            "gene_list",
        ]
    )
    .reset_index(
        drop=True
    )
)


observed_sizes = dict(
    zip(
        inventory[
            "gene_list"
        ],
        inventory[
            "n_genes"
        ],
    )
)


if observed_sizes != EXPECTED_HISTORICAL_LIST_SIZES:
    expected_keys = set(
        EXPECTED_HISTORICAL_LIST_SIZES
    )

    observed_keys = set(
        observed_sizes
    )

    mismatch_rows = []

    for name in sorted(
        expected_keys
        | observed_keys
    ):
        expected = EXPECTED_HISTORICAL_LIST_SIZES.get(
            name
        )

        observed = observed_sizes.get(
            name
        )

        if expected != observed:
            mismatch_rows.append(
                {
                    "gene_list": name,
                    "expected_n_genes": expected,
                    "observed_n_genes": observed,
                }
            )

    display(
        pd.DataFrame(
            mismatch_rows
        )
    )

    raise RuntimeError(
        "Historical g:Profiler directional gene-list inventory checkpoint failed."
    )


write_tsv(
    inventory,
    METADATA_DIR
    / f"{OUT_PREFIX}_gene_list_inventory.tsv",
)


readiness = pd.DataFrame(
    [
        {
            "item": "state_marker_DE_file",
            "path": str(
                STATE_SIG_TSV
            ),
            "exists": STATE_SIG_TSV.exists(),
            "n_rows": len(
                state_sig
            ),
        },
        {
            "item": "condition_overall_DE_file",
            "path": str(
                COND_SIG_TSV
            ),
            "exists": COND_SIG_TSV.exists(),
            "n_rows": len(
                cond_sig
            ),
        },
        {
            "item": "historical_directional_gene_lists",
            "path": "in_memory",
            "exists": True,
            "n_rows": len(
                gene_lists
            ),
        },
        {
            "item": "gprofiler_api",
            "path": GPROFILER_URL,
            "exists": True,
            "n_rows": np.nan,
        },
    ]
)


write_tsv(
    readiness,
    METADATA_DIR
    / f"{OUT_PREFIX}_readiness.tsv",
)


display(
    inventory
)


print(
    "PASS: exact historical directional inventory reproduced."
)

print(
    "Gene lists prepared:",
    len(
        gene_lists
    ),
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\online_enrichment_validation_HVG8000_r0_6_gprofiler_only_gene_list_inventory.tsv shape=(17, 5)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\online_enrichment_validation_HVG8000_r0_6_gprofiler_only_readiness.tsv shape=(4, 4)


,gene_list,source,group,direction,n_genes
0,condition_overall__GBM_vs_LGG__GBM_up,condition_overall_malignant_DE,GBM_vs_LGG,GBM_up,300
1,condition_overall__GBM_vs_LGG__LGG_up,condition_overall_malignant_DE,GBM_vs_LGG,LGG_up,300
2,condition_overall__GBM_vs_ndGBM__GBM_up,condition_overall_malignant_DE,GBM_vs_ndGBM,GBM_up,300
3,condition_overall__GBM_vs_ndGBM__ndGBM_up,condition_overall_malignant_DE,GBM_vs_ndGBM,ndGBM_up,300
4,condition_overall__GBM_vs_rGBM__GBM_up,condition_overall_malignant_DE,GBM_vs_rGBM,GBM_up,300
5,condition_overall__GBM_vs_rGBM__rGBM_up,condition_overall_malignant_DE,GBM_vs_rGBM,rGBM_up,300
6,condition_overall__LGG_vs_ndGBM__LGG_up,condition_overall_malignant_DE,LGG_vs_ndGBM,LGG_up,300
7,condition_overall__LGG_vs_ndGBM__ndGBM_up,condition_overall_malignant_DE,LGG_vs_ndGBM,ndGBM_up,300
8,condition_overall__LGG_vs_rGBM__LGG_up,condition_overall_malignant_DE,LGG_vs_rGBM,LGG_up,300
9,condition_overall__LGG_vs_rGBM__rGBM_up,condition_overall_malignant_DE,LGG_vs_rGBM,rGBM_up,300


PASS: exact historical directional inventory reproduced.
Gene lists prepared: 17


In [3]:
# =========================
# 2. Run current g:Profiler API snapshot
# =========================
def post_json_with_retries(
    url,
    payload,
):
    encoded = json.dumps(
        payload
    ).encode(
        "utf-8"
    )

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):
        request = urllib.request.Request(
            url,
            data=encoded,
            headers={
                "Content-Type": "application/json",
                "User-Agent": "glioma-cnv-single-cell-atlas/1.0",
            },
        )

        try:
            with urllib.request.urlopen(
                request,
                timeout=REQUEST_TIMEOUT_SEC,
            ) as response:
                body = response.read().decode(
                    "utf-8"
                )

            return json.loads(
                body
            )

        except Exception as exc:
            last_error = exc

            if attempt < MAX_RETRIES:
                time.sleep(
                    2 ** (
                        attempt - 1
                    )
                )

    raise RuntimeError(
        f"g:Profiler request failed after {MAX_RETRIES} attempts: "
        f"{last_error!r}"
    )


def run_gprofiler(
    list_name,
    genes,
):
    payload = {
        "organism": "hsapiens",
        "query": genes,
        "sources": GPROFILER_SOURCES,
        "user_threshold": 0.05,
        "all_results": False,
        "no_evidences": True,
        "ordered": False,
        "significance_threshold_method": "g_SCS",
    }

    requested_utc = datetime.now(
        timezone.utc
    ).isoformat()

    response = post_json_with_retries(
        GPROFILER_URL,
        payload,
    )

    rows = []

    for result in response.get(
        "result",
        [],
    ):
        p_value = result.get(
            "p_value"
        )

        rows.append(
            {
                "gene_list": list_name,
                "source": result.get(
                    "source"
                ),
                "term_id": result.get(
                    "native"
                ),
                "term_name": result.get(
                    "name"
                ),
                "p_value": p_value,
                "negative_log10_p": (
                    -math.log10(
                        p_value
                    )
                    if (
                        p_value
                        and p_value > 0
                    )
                    else np.inf
                ),
                "term_size": result.get(
                    "term_size"
                ),
                "query_size": result.get(
                    "query_size"
                ),
                "intersection_size": result.get(
                    "intersection_size"
                ),
                "precision": result.get(
                    "precision"
                ),
                "recall": result.get(
                    "recall"
                ),
                "intersections": ";".join(
                    result.get(
                        "intersections",
                        [],
                    )
                    or []
                ),
            }
        )

    metadata_row = {
        "gene_list": list_name,
        "n_query_genes": len(
            genes
        ),
        "requested_utc": requested_utc,
        "api_url": GPROFILER_URL,
        "sources": ";".join(
            GPROFILER_SOURCES
        ),
        "organism": "hsapiens",
        "user_threshold": 0.05,
        "ordered": False,
        "significance_threshold_method": "g_SCS",
        "n_result_rows": len(
            rows
        ),
        "response_meta_json": json.dumps(
            response.get(
                "meta",
                {},
            ),
            sort_keys=True,
            ensure_ascii=False,
        ),
    }

    return (
        rows,
        metadata_row,
    )


result_rows = []
error_rows = []
api_metadata_rows = []


for number, (
    name,
    genes,
) in enumerate(
    gene_lists.items(),
    start=1,
):
    print(
        f"[g:Profiler] {number}/{len(gene_lists)} "
        f"{name} n={len(genes)}"
    )

    try:
        rows, metadata_row = run_gprofiler(
            name,
            genes,
        )

        result_rows.extend(
            rows
        )

        api_metadata_rows.append(
            metadata_row
        )

    except Exception as exc:
        error_rows.append(
            {
                "gene_list": name,
                "n_genes": len(
                    genes
                ),
                "error": repr(
                    exc
                ),
            }
        )

        print(
            "[ERROR]",
            name,
            repr(
                exc
            ),
        )

    time.sleep(
        REQUEST_SLEEP_SEC
    )


result_columns = [
    "gene_list",
    "source",
    "term_id",
    "term_name",
    "p_value",
    "negative_log10_p",
    "term_size",
    "query_size",
    "intersection_size",
    "precision",
    "recall",
    "intersections",
]


results = pd.DataFrame(
    result_rows,
    columns=result_columns,
)


errors = pd.DataFrame(
    error_rows,
    columns=[
        "gene_list",
        "n_genes",
        "error",
    ],
)


api_metadata = pd.DataFrame(
    api_metadata_rows
)


if len(
    results
):
    top_terms = (
        results
        .sort_values(
            [
                "gene_list",
                "p_value",
            ]
        )
        .groupby(
            "gene_list",
            group_keys=False,
        )
        .head(
            10
        )
        .reset_index(
            drop=True
        )
    )

else:
    top_terms = results.copy()


write_tsv(
    results,
    METADATA_DIR
    / f"{OUT_PREFIX}_gprofiler_results.tsv",
)


write_tsv(
    top_terms,
    METADATA_DIR
    / f"{OUT_PREFIX}_gprofiler_top_terms.tsv",
)


write_tsv(
    errors,
    METADATA_DIR
    / f"{OUT_PREFIX}_gprofiler_errors.tsv",
)


write_tsv(
    api_metadata,
    METADATA_DIR
    / f"{OUT_PREFIX}_gprofiler_api_metadata.tsv",
)


print(
    "Current API result rows:",
    len(
        results
    ),
)

print(
    "Current API top-term rows:",
    len(
        top_terms
    ),
)

print(
    "Current API errors:",
    len(
        errors
    ),
)

display(
    results.head(
        20
    )
)

display(
    errors
)


[g:Profiler] 1/17 state_marker__AC_like_reactive_malignant__up n=300
[g:Profiler] 2/17 state_marker__NPC_neuronal_like_malignant__up n=300
[g:Profiler] 3/17 state_marker__OPC_NPC_like_malignant__up n=300
[g:Profiler] 4/17 state_marker__cycling_malignant_candidate__up n=300
[g:Profiler] 5/17 state_marker__hypoxia_stress_malignant__up n=300
[g:Profiler] 6/17 condition_overall__GBM_vs_LGG__GBM_up n=300
[g:Profiler] 7/17 condition_overall__GBM_vs_LGG__LGG_up n=300
[g:Profiler] 8/17 condition_overall__GBM_vs_ndGBM__GBM_up n=300
[g:Profiler] 9/17 condition_overall__GBM_vs_ndGBM__ndGBM_up n=300
[g:Profiler] 10/17 condition_overall__GBM_vs_rGBM__GBM_up n=300
[g:Profiler] 11/17 condition_overall__GBM_vs_rGBM__rGBM_up n=300
[g:Profiler] 12/17 condition_overall__LGG_vs_ndGBM__LGG_up n=300
[g:Profiler] 13/17 condition_overall__LGG_vs_ndGBM__ndGBM_up n=300
[g:Profiler] 14/17 condition_overall__LGG_vs_rGBM__LGG_up n=300
[g:Profiler] 15/17 condition_overall__LGG_vs_rGBM__rGBM_up n=300
[g:Profiler] 16

,gene_list,source,term_id,term_name,p_value,negative_log10_p,term_size,query_size,intersection_size,precision,recall,intersections
0,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0050896,response to stimulus,2.700004e-09,8.568636,9051,274,176,0.642336,0.019445,
1,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0048856,anatomical structure development,1.418248e-07,6.848248,5948,274,128,0.467153,0.021520,
2,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0032502,developmental process,1.122317e-06,5.949885,6535,274,134,0.489051,0.020505,
3,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0009653,anatomical structure morphogenesis,1.236397e-06,5.907842,2699,274,73,0.266423,0.027047,
4,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0032501,multicellular organismal process,1.535227e-06,5.813827,7337,274,145,0.529197,0.019763,
5,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0007155,cell adhesion,2.020803e-06,5.694476,1512,274,50,0.182482,0.033069,
6,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0007399,nervous system development,3.682909e-06,5.433809,2597,274,70,0.255474,0.026954,
7,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0051716,cellular response to stimulus,3.863104e-06,5.413064,7425,274,145,0.529197,0.019529,
8,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0051179,localization,8.411788e-06,5.075112,5493,274,116,0.423358,0.021118,
9,state_marker__AC_like_reactive_malignant__up,GO:BP,GO:0007154,cell communication,9.959886e-06,5.001746,6598,274,132,0.481752,0.020006,


,gene_list,n_genes,error


In [4]:
# =========================
# 3. Preserve historical biological keyword categorization
# =========================
CATEGORY_KEYWORDS = {
    "hypoxia_glycolysis_stress": [
        "hypoxia",
        "hif",
        "glycolysis",
        "glucose",
        "lactate",
        "oxidative stress",
        "unfolded protein",
        "stress",
    ],
    "cycling_E2F_G2M": [
        "cell cycle",
        "mitotic",
        "mitosis",
        "g2m",
        "e2f",
        "dna replication",
        "chromosome segregation",
        "spindle",
    ],
    "MES_ECM_EMT_invasion": [
        "epithelial mesenchymal",
        "emt",
        "extracellular matrix",
        "collagen",
        "matrix",
        "focal adhesion",
        "migration",
        "invasion",
        "wound",
    ],
    "OPC_oligodendrocyte_glial": [
        "oligodendrocyte",
        "glial",
        "gliogenesis",
        "myelination",
        "axon ensheathment",
    ],
    "neural_NPC_development": [
        "neuron",
        "neural",
        "neurogenesis",
        "axon",
        "synapse",
        "dendrite",
        "forebrain",
        "nervous system development",
    ],
    "immune_inflammation_interferon_TNF": [
        "immune",
        "inflammatory",
        "interferon",
        "cytokine",
        "tnf",
        "nf-kappa",
        "leukocyte",
        "antigen",
    ],
    "angiogenesis_endothelial": [
        "angiogenesis",
        "endothelial",
        "blood vessel",
        "vasculature",
        "vascular",
    ],
    "dna_damage_repair": [
        "dna repair",
        "dna damage",
        "double-strand",
        "replication stress",
        "checkpoint",
    ],
}


def categorize_term(term):
    text = str(
        term
    ).lower()

    categories = []

    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(
            keyword in text
            for keyword
            in keywords
        ):
            categories.append(
                category
            )

    return (
        ";".join(
            categories
        )
        if categories
        else "other"
    )


summary_rows = []


if len(
    results
):
    categorized = results.copy()

    categorized[
        "category"
    ] = categorized[
        "term_name"
    ].map(
        categorize_term
    )

    categorized = categorized.loc[
        categorized[
            "category"
        ]
        .ne(
            "other"
        )
    ]

    for (
        gene_list,
        category,
    ), sub in categorized.groupby(
        [
            "gene_list",
            "category",
        ]
    ):
        summary_rows.append(
            {
                "source": "gProfiler",
                "gene_list": gene_list,
                "category": category,
                "n_terms": len(
                    sub
                ),
                "best_p_value": pd.to_numeric(
                    sub[
                        "p_value"
                    ],
                    errors="coerce",
                ).min(),
                "top_terms": " | ".join(
                    sub
                    .sort_values(
                        "p_value"
                    )[
                        "term_name"
                    ]
                    .head(
                        5
                    )
                    .astype(str)
                ),
            }
        )


biological_summary = pd.DataFrame(
    summary_rows,
    columns=[
        "source",
        "gene_list",
        "category",
        "n_terms",
        "best_p_value",
        "top_terms",
    ],
)


if len(
    biological_summary
):
    biological_summary = (
        biological_summary
        .sort_values(
            [
                "gene_list",
                "source",
                "best_p_value",
            ]
        )
        .reset_index(
            drop=True
        )
    )


if len(
    biological_summary
):
    category_counts = (
        biological_summary
        .groupby(
            [
                "source",
                "category",
            ]
        )
        .size()
        .reset_index(
            name="n_gene_list_category_hits"
        )
    )

else:
    category_counts = pd.DataFrame(
        columns=[
            "source",
            "category",
            "n_gene_list_category_hits",
        ]
    )


write_tsv(
    biological_summary,
    METADATA_DIR
    / f"{OUT_PREFIX}_biological_keyword_summary.tsv",
)


write_tsv(
    category_counts,
    METADATA_DIR
    / f"{OUT_PREFIX}_biological_category_counts.tsv",
)


display(
    biological_summary.head(
        60
    )
)


display(
    category_counts
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\online_enrichment_validation_HVG8000_r0_6_gprofiler_only_biological_keyword_summary.tsv shape=(46, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\online_enrichment_validation_HVG8000_r0_6_gprofiler_only_biological_category_counts.tsv shape=(15, 3)


,source,gene_list,category,n_terms,best_p_value,top_terms
0,gProfiler,condition_overall__GBM_vs_LGG__GBM_up,immune_inflammation_interferon_TNF,11,6.568603e-05,immune system process | Immune System | Antige...
1,gProfiler,condition_overall__GBM_vs_LGG__GBM_up,MES_ECM_EMT_invasion,4,4.515816e-04,Extracellular matrix organization | regulation...
2,gProfiler,condition_overall__GBM_vs_LGG__GBM_up,angiogenesis_endothelial,3,4.987433e-03,vasculature development | blood vessel develop...
3,gProfiler,condition_overall__GBM_vs_LGG__LGG_up,OPC_oligodendrocyte_glial,1,2.142391e-02,Oligodendrocyte development
4,gProfiler,condition_overall__GBM_vs_ndGBM__ndGBM_up,dna_damage_repair,1,2.665028e-02,DNA repair
5,gProfiler,condition_overall__GBM_vs_rGBM__GBM_up,immune_inflammation_interferon_TNF,8,1.638924e-03,"Antigen Presentation: Folding, assembly and pe..."
6,gProfiler,condition_overall__GBM_vs_rGBM__rGBM_up,cycling_E2F_G2M,15,5.844737e-06,cell cycle process | cell cycle | mitotic cell...
7,gProfiler,condition_overall__GBM_vs_rGBM__rGBM_up,dna_damage_repair,4,4.491422e-05,DNA repair | DNA damage response | double-stra...
8,gProfiler,condition_overall__GBM_vs_rGBM__rGBM_up,hypoxia_glycolysis_stress,1,1.741307e-04,cellular response to stress
9,gProfiler,condition_overall__LGG_vs_ndGBM__LGG_up,neural_NPC_development,11,9.382226e-14,nervous system development | neurogenesis | ge...


,source,category,n_gene_list_category_hits
0,gProfiler,MES_ECM_EMT_invasion,5
1,gProfiler,MES_ECM_EMT_invasion;immune_inflammation_inter...,2
2,gProfiler,MES_ECM_EMT_invasion;neural_NPC_development,1
3,gProfiler,OPC_oligodendrocyte_glial,4
4,gProfiler,OPC_oligodendrocyte_glial;neural_NPC_development,1
5,gProfiler,angiogenesis_endothelial,4
6,gProfiler,cycling_E2F_G2M,3
7,gProfiler,cycling_E2F_G2M;dna_damage_repair,1
8,gProfiler,cycling_E2F_G2M;immune_inflammation_interferon...,1
9,gProfiler,dna_damage_repair,4


In [5]:
# =========================
# 4. Historical-vs-current API snapshot audit and output manifest
# =========================
historical_reference = pd.DataFrame(
    [
        {
            "checkpoint": key,
            "historical_value": value,
        }
        for key, value
        in HISTORICAL_API_SNAPSHOT.items()
    ]
)


write_tsv(
    historical_reference,
    METADATA_DIR
    / f"{OUT_PREFIX}_historical_snapshot_reference.tsv",
)


successful_current_lists = (
    results[
        "gene_list"
    ]
    .nunique()
    if len(
        results
    )
    else 0
)


current_snapshot = pd.DataFrame(
    [
        {
            "metric": "submitted_gene_lists",
            "historical": HISTORICAL_API_SNAPSHOT[
                "submitted_gene_lists"
            ],
            "current": len(
                gene_lists
            ),
            "interpretation": "deterministic input checkpoint",
        },
        {
            "metric": "successful_gene_lists",
            "historical": HISTORICAL_API_SNAPSHOT[
                "successful_gene_lists"
            ],
            "current": successful_current_lists,
            "interpretation": "online API coverage; may differ because the historical run had a timeout",
        },
        {
            "metric": "gprofiler_result_rows",
            "historical": HISTORICAL_API_SNAPSHOT[
                "gprofiler_result_rows"
            ],
            "current": len(
                results
            ),
            "interpretation": "online database snapshot; exact equality is not expected",
        },
        {
            "metric": "gprofiler_top_term_rows",
            "historical": HISTORICAL_API_SNAPSHOT[
                "gprofiler_top_term_rows"
            ],
            "current": len(
                top_terms
            ),
            "interpretation": "online database snapshot; exact equality is not expected",
        },
        {
            "metric": "gprofiler_error_rows",
            "historical": HISTORICAL_API_SNAPSHOT[
                "gprofiler_error_rows"
            ],
            "current": len(
                errors
            ),
            "interpretation": "transient network/API behavior",
        },
        {
            "metric": "biological_keyword_summary_rows",
            "historical": HISTORICAL_API_SNAPSHOT[
                "biological_keyword_summary_rows"
            ],
            "current": len(
                biological_summary
            ),
            "interpretation": "depends on current g:Profiler term snapshot",
        },
        {
            "metric": "biological_category_count_rows",
            "historical": HISTORICAL_API_SNAPSHOT[
                "biological_category_count_rows"
            ],
            "current": len(
                category_counts
            ),
            "interpretation": "depends on current g:Profiler term snapshot",
        },
    ]
)


write_tsv(
    current_snapshot,
    METADATA_DIR
    / f"{OUT_PREFIX}_historical_vs_current_snapshot_audit.tsv",
)


output_names = [
    f"{OUT_PREFIX}_readiness.tsv",
    f"{OUT_PREFIX}_gene_list_inventory.tsv",
    f"{OUT_PREFIX}_gprofiler_results.tsv",
    f"{OUT_PREFIX}_gprofiler_top_terms.tsv",
    f"{OUT_PREFIX}_gprofiler_errors.tsv",
    f"{OUT_PREFIX}_gprofiler_api_metadata.tsv",
    f"{OUT_PREFIX}_biological_keyword_summary.tsv",
    f"{OUT_PREFIX}_biological_category_counts.tsv",
    f"{OUT_PREFIX}_historical_snapshot_reference.tsv",
    f"{OUT_PREFIX}_historical_vs_current_snapshot_audit.tsv",
]


manifest = pd.DataFrame(
    [
        {
            "file": name,
            "path": str(
                METADATA_DIR
                / name
            ),
            "exists": (
                METADATA_DIR
                / name
            ).exists(),
            "size_bytes": (
                (
                    METADATA_DIR
                    / name
                ).stat().st_size
                if (
                    METADATA_DIR
                    / name
                ).exists()
                else 0
            ),
        }
        for name
        in output_names
    ]
)


write_tsv(
    manifest,
    METADATA_DIR
    / f"{OUT_PREFIX}_output_manifest.tsv",
)


if not manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more Notebook-24 outputs are missing."
    )


display(
    current_snapshot
)


display(
    manifest
)


print(
    "\nFINAL CHECKPOINTS"
)

print(
    "Historical notebook mapping:",
    "22b_online_enrichment_validation_HVG8000_r0_6_v2_no_pip_direct_api.ipynb "
    "-> 24_gprofiler_validation.ipynb",
)

print(
    "Historical g:Profiler directionality:",
    "already correct; preserved",
)

print(
    "State-DE rows:",
    len(
        state_sig
    ),
)

print(
    "Condition-DE rows:",
    len(
        cond_sig
    ),
)

print(
    "Directional gene-list inventory:",
    inventory.shape,
)

print(
    "Submitted gene lists:",
    len(
        gene_lists
    ),
)

print(
    "Current successful gene lists:",
    successful_current_lists,
)

print(
    "Current g:Profiler results:",
    results.shape,
)

print(
    "Current g:Profiler top terms:",
    top_terms.shape,
)

print(
    "Current g:Profiler errors:",
    errors.shape,
)

print(
    "Current biological keyword summary:",
    biological_summary.shape,
)

print(
    "Current biological category counts:",
    category_counts.shape,
)

print(
    "Output manifest:",
    manifest.shape,
)


if len(
    errors
):
    print(
        "\n[WARNING] One or more live g:Profiler requests failed after retries. "
        "The failures were recorded. Because this is an online validation step, "
        "rerunning the API cell later is appropriate if complete current coverage is required."
    )


print(
    "\n[DONE] Notebook 24 g:Profiler validation completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\online_enrichment_validation_HVG8000_r0_6_gprofiler_only_historical_snapshot_reference.tsv shape=(9, 2)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\online_enrichment_validation_HVG8000_r0_6_gprofiler_only_historical_vs_current_snapshot_audit.tsv shape=(7, 4)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\online_enrichment_validation_HVG8000_r0_6_gprofiler_only_output_manifest.tsv shape=(10, 4)


,metric,historical,current,interpretation
0,submitted_gene_lists,17,17,deterministic input checkpoint
1,successful_gene_lists,16,17,online API coverage; may differ because the hi...
2,gprofiler_result_rows,1669,1738,online database snapshot; exact equality is no...
3,gprofiler_top_term_rows,160,170,online database snapshot; exact equality is no...
4,gprofiler_error_rows,1,0,transient network/API behavior
5,biological_keyword_summary_rows,43,46,depends on current g:Profiler term snapshot
6,biological_category_count_rows,15,15,depends on current g:Profiler term snapshot


,file,path,exists,size_bytes
0,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,514
1,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,1722
2,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,325685
3,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,31150
4,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,25
5,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,369997
6,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,9460
7,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,758
8,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,371
9,online_enrichment_validation_HVG8000_r0_6_gpro...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,605



FINAL CHECKPOINTS
Historical notebook mapping: 22b_online_enrichment_validation_HVG8000_r0_6_v2_no_pip_direct_api.ipynb -> 24_gprofiler_validation.ipynb
Historical g:Profiler directionality: already correct; preserved
State-DE rows: 20800
Condition-DE rows: 27004
Directional gene-list inventory: (17, 5)
Submitted gene lists: 17
Current successful gene lists: 17
Current g:Profiler results: (1738, 12)
Current g:Profiler top terms: (170, 12)
Current g:Profiler errors: (0, 3)
Current biological keyword summary: (46, 6)
Current biological category counts: (15, 3)
Output manifest: (10, 4)

[DONE] Notebook 24 g:Profiler validation completed.


## Interpretation

The deterministic reproducibility target in this notebook is the **directional input
gene-list construction**, not exact present-day g:Profiler term counts.

The historical g:Profiler branch already used the correct direction:

- state lists were `up` genes only;
- each condition contrast was split into the two explicit expression directions.

Therefore, Notebook 23's curated-ORA direction-detection correction does not imply a
corresponding directionality correction here.

Any differences between the historical and current g:Profiler result tables should be
evaluated as either:

1. transient API/network behavior, or
2. annotation-database snapshot differences,

rather than being forced to match by modifying the input gene lists.